# P31 training on Kaggle

1. Settings → Accelerator: **GPU T4 x2** or **P100**, Internet: **on**.
2. Add data → search **msambare/fer2013** (appears at `/kaggle/input/fer2013`).
3. Optional: upload RAF-DB as a private Kaggle dataset named `rafdb` for the cross-dataset test.
4. Set `REPO_URL` and `MODEL` below, then Run All. Download `results/` from the Output tab.

In [ ]:
REPO_URL = "https://github.com/Neeraj-NP/P31-facial-expression-recognition.git"
MODEL = "cnn"   # cnn | resnet | vit

# Kaggle mounts inputs at /kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>,
# so locate each dataset by a folder it must contain instead of hard-coding the path.
from pathlib import Path
def find_root(marker, levels_up, base="/kaggle/input"):
    hits = sorted(Path(base).rglob(marker))
    return str(hits[0].parents[levels_up - 1]) if hits else None

FER_ROOT = find_root("train/happy", 2)    # <root>/train/happy
RAF_ROOT = find_root("EmoLabel", 1) or "/kaggle/input/rafdb"   # <root>/EmoLabel; ignored if not attached
assert FER_ROOT, "FER-2013 not found: Add Input -> https://www.kaggle.com/datasets/msambare/fer2013"
print("FER_ROOT =", FER_ROOT, "| RAF_ROOT =", RAF_ROOT)

In [ ]:
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q timm grad-cam
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# point the config at Kaggle's read-only input folders
import yaml
cfg = yaml.safe_load(open(f"configs/{MODEL}.yaml"))
cfg["data"].update(fer2013_root=FER_ROOT, rafdb_root=RAF_ROOT, num_workers=2)
yaml.safe_dump(cfg, open(f"configs/{MODEL}_kaggle.yaml", "w"))
cfg

In [ ]:
!python -m src.train --config configs/{MODEL}_kaggle.yaml --run-name {MODEL}

In [ ]:
!python -m src.evaluate --run results/{MODEL} --dataset fer2013
!python -m src.explain --run results/{MODEL} --dataset fer2013

In [ ]:
# cross-dataset test + subgroup audit (needs RAF-DB attached)
import os
if os.path.exists(RAF_ROOT):
    !python -m src.evaluate --run results/{MODEL} --dataset rafdb
    !python -m src.explain --run results/{MODEL} --dataset rafdb --group-by gender
else:
    print("RAF-DB not attached, skipping")

In [ ]:
!cd /kaggle/working && zip -qr results_{MODEL}.zip repo/results/{MODEL}
print("download results_" + MODEL + ".zip from the Output tab")